# Multiple Linear Regression - Sunoco Gasoline Demand

This notebook uses multiple linear regression to estimate weekly gasoline demand from a set
of economic and energy market predictors. Multiple linear regression works the same way as
simple linear regression, except that we pass more than one column to the X argument. The
output gives us one coefficient per predictor plus an intercept, and we can plug those
coefficients into an equation to calculate an estimate of demand for any row of data.

This version runs on the extended dataset, which carries the target through the end of 2024
instead of stopping in 2019. That matters more than it sounds: the added years contain the
2020 pandemic, and gasoline demand never returned to its old level afterwards. A good part
of this notebook is about detecting that and deciding what to do about it, because a model
fitted straight through the break will quietly mislead you.

We check three things a single r2 will not tell us:

- whether each model is fitted on the same rows, so the fit statistics are comparable,
- whether predictors are near-duplicates of each other, which leaves the overall fit intact
  while making individual coefficients meaningless, and
- whether the relationship is stable over time, or whether it broke partway through.

In [ ]:
import pandas as pd
import pingouin as pg
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import itertools

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

DATA_FILE = 'Sunoco Training_complete_NEW.xlsx'

## Read in the data

The demand values live on the Target sheet. The training data runs from the first row until
Demand stops being populated; everything after that is the holdout period we are eventually
asked to forecast.

Rather than hard-coding the boundary row, we derive it from where Demand ends and then
assert the slice is clean. That way the slice cannot silently drift out of alignment with
the sheet if the file is extended again.

In [ ]:
sunoco = pd.read_excel(DATA_FILE, sheet_name = 'Target')
sunoco['Date'] = pd.to_datetime(sunoco['Date'])
print("full sheet:", sunoco.shape)

sunoco.head()

In [ ]:
last_valid_idx = sunoco['Demand'].last_valid_index()
training = sunoco.iloc[:last_valid_idx + 1].copy()

# Integrity assertions
assert training['Demand'].notna().all(), "training slice contains a week with no Demand"
assert sunoco['Demand'].iloc[last_valid_idx + 1:].isna().all(), "Demand reappears after the slice"

print(len(training), "training weeks")
print("training :", training['Date'].min().date(), "to", training['Date'].max().date())
print("holdout  :", sunoco['Date'].iloc[last_valid_idx + 1].date(), "to", sunoco['Date'].max().date(),
      f"({len(sunoco) - (last_valid_idx + 1)} weeks)")

## Look at the target before modelling anything

Plotting the series first is worth the one cell it costs. Summary statistics will not tell
you that the process changed partway through, and every regression below assumes it did not.

In [ ]:
fig, ax = plt.subplots(figsize = (13, 5))
ax.plot(training.Date, training.Demand, linewidth = 0.8, color = "#2b6cb0")
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-01-01"),
           color = "crimson", alpha = 0.12, label = "Mar-Dec 2020")
ax.axhline(training[training.Date < "2020-03-01"].Demand.mean(),
           color = "grey", linestyle = "--", linewidth = 1, label = "pre-2020 mean")
ax.axhline(training[training.Date >= "2021-01-01"].Demand.mean(),
           color = "darkorange", linestyle = "--", linewidth = 1, label = "2021+ mean")
ax.set_title("Weekly gasoline demand, 1991-2024")
ax.set_ylabel("Demand")
ax.legend()
plt.show()

print(training.nsmallest(4, "Demand")[["Date", "Demand", "vehicle_miles_mil",
                                       "initial_jobless_claims"]].to_string(index = False))

Two features stand out, and neither existed in the old 1991-2019 version of this file.

**A collapse in spring 2020.** Demand bottomed at 5,065 in the week of 3 April 2020, about
45% below the surrounding trend, with jobless claims at 5.9 million in the same week. That
is far outside anything else in thirty years of data.

**A level shift that never reversed.** The pre-2020 mean is roughly 9,000; from 2021 onward
it is roughly 8,800, and demand has not returned to its old range even as miles driven and
population recovered. Whatever relationship held before 2020 does not obviously still hold.

Keep both in mind. We will fit the standard models first, exactly as before, and then test
formally whether pooling across the break is defensible.

## Explore the predictors

Before building a model, let's look at how each predictor correlates with demand. A strong
correlation in either direction suggests the variable is worth including.

In [ ]:
predictors = ['vehicle_miles_mil', 'us_population_thous', 'gdp_saar_bil_usd',
              'regular_gas_usd_per_gal', 'wti_spot_usd_per_bbl',
              'brent_spot_usd_per_bbl', 'refinery_utilization_pct',
              'initial_jobless_claims']

training[['Demand'] + predictors].corr()['Demand'].sort_values()

In [ ]:
plt.figure(figsize = (9, 7))
sns.heatmap(training[['Demand'] + predictors].corr(), annot = True, fmt = ".2f", cmap = "Blues")
plt.title("Correlation among Demand and candidate predictors")
plt.show()

Vehicle miles is still the strongest single predictor at 0.70, followed by population at
0.59 and gas price at 0.56. Every one of these is weaker than it was on the 1991-2019
sample, where vehicle miles reached 0.80 and population 0.77. The extra five years have made
the relationships look worse, which is our first quantitative hint of the break.

Now the more important part of that heatmap, which is what happens *off* the Demand row.
Rather than eyeballing it, let's list every predictor pair above 0.95.

In [ ]:
corr_matrix = training[predictors].corr().abs()
pairs = (corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k = 1).astype(bool))
                    .stack()
                    .sort_values(ascending = False))

print("Predictor pairs correlated above 0.95:")
print(pairs[pairs > 0.95].round(4).to_string())

Four near-duplicate pairs, built from two clusters:

- **A price cluster.** wti and brent at 0.99, and regular gas price against each of them at
  about 0.96-0.97. Three variables all tracking the price of crude.
- **A trend cluster.** GDP and population at 0.96. Both are smooth series that rise nearly
  monotonically, so over this sample they carry almost the same information.

That is multicollinearity, and it will cause trouble in any model containing more than one
member of a cluster. It does not hurt the overall fit; it makes the individual coefficients
unreliable and often flips their signs. Watch for it in Models B and D.

Based on this we can state our hypotheses:

H0 - these predictors do not influence weekly gasoline demand <br>
H1 - these predictors do influence weekly gasoline demand

## Helpers

Every model below is scored the same way, so let's write it once. `evaluate` returns `n`
alongside the error statistics, because each model drops a different set of rows for missing
values and fit statistics are only comparable across models fitted on the same rows.

In [ ]:
def evaluate(data, actual_col = "Demand", calculated_col = "calculated_demand"):
    error = data[actual_col] - data[calculated_col]
    return {"n": len(data),
            "RMSE": np.sqrt(np.mean(error ** 2)),
            "MAE": np.mean(np.abs(error))}


def show_vif(data, preds):
    X = sm.add_constant(data[preds])
    return pd.DataFrame({
        "predictor": preds,
        "VIF": [variance_inflation_factor(X.values, X.columns.get_loc(p)) for p in preds]
    }).sort_values("VIF", ascending = False).round(1)

## Model A - two predictors

We have to drop rows with missing values first, because `linear_regression` cannot handle
blanks. `us_population_thous` is missing for 56 weeks in 1991-92, which is why we lose a few
rows.

In [ ]:
model_a_predictors = ['vehicle_miles_mil', 'us_population_thous']
model_a_data = training[['Demand', 'Date'] + model_a_predictors].dropna().copy()
print(len(model_a_data), "rows of", len(training))

model_a = pg.linear_regression(X = model_a_data[model_a_predictors], y = model_a_data.Demand)
print(model_a)

Both predictors are significant, but the model explains only about 46% of the variation in
demand. On the 1991-2019 data this same model reached 67%. Nothing about the specification
changed; the extra five years did that.

Now let's use the coefficients to calculate the model's estimate for each week, exactly the
way we did with mtcars. `model_a.coef[0]` is the intercept and the slopes follow in the same
order as the columns passed to X.

In [ ]:
model_a_data["calculated_demand"] = (model_a.coef[0]
                                     + model_a.coef[1] * model_a_data.vehicle_miles_mil
                                     + model_a.coef[2] * model_a_data.us_population_thous)

score_a = evaluate(model_a_data)
score_a

Two error measures, answering slightly different questions.

**MAE (about 386)** is the average size of the miss - the number you can describe as "in a
typical week the model is off by about 386 units."

**RMSE (about 509)** squares the errors before averaging, so a handful of large misses pull
it upward. It is always greater than or equal to MAE, and the gap between them tells you how
lumpy the errors are. RMSE is the standard number to report because it is what least squares
actually minimises, but it is not an average error, so do not describe it as one.

## Model B - three predictors

Adding GDP.

In [ ]:
model_b_predictors = ['vehicle_miles_mil', 'gdp_saar_bil_usd', 'us_population_thous']
model_b_data = training[['Demand', 'Date'] + model_b_predictors].dropna().copy()

model_b = pg.linear_regression(X = model_b_data[model_b_predictors], y = model_b_data.Demand)
print(model_b)

In [ ]:
model_b_data["calculated_demand"] = (model_b.coef[0]
                                     + model_b.coef[1] * model_b_data.vehicle_miles_mil
                                     + model_b.coef[2] * model_b_data.gdp_saar_bil_usd
                                     + model_b.coef[3] * model_b_data.us_population_thous)

score_b = evaluate(model_b_data)
score_b

The fit improves, but check the GDP coefficient: it is **negative** and strongly
significant. Read literally that says a growing economy reduces gasoline consumption, which
contradicts both economic sense and the simple correlation we computed above (+0.51).

This is the GDP/population collinearity. The two are 0.96 correlated, so the regression can
only identify their *difference*, and it splits them into a large positive population effect
and an offsetting negative GDP effect. Variance inflation factors confirm it - anything above
roughly 5 to 10 is a warning sign.

In [ ]:
show_vif(model_b_data, model_b_predictors)

## Model C - five predictors

The variables the hypotheses sheet expects to matter: gas price, driving activity, refinery
utilization, population, and jobless claims. Note that it deliberately includes only one
member of each correlated cluster.

In [ ]:
model_c_predictors = ['regular_gas_usd_per_gal', 'vehicle_miles_mil',
                      'refinery_utilization_pct', 'us_population_thous',
                      'initial_jobless_claims']

model_c_data = training[['Demand', 'Date'] + model_c_predictors].dropna().copy()

model_c = pg.linear_regression(X = model_c_data[model_c_predictors], y = model_c_data.Demand)
print(model_c)

In [ ]:
model_c_data["calculated_demand"] = (model_c.coef[0]
                                     + model_c.coef[1] * model_c_data.regular_gas_usd_per_gal
                                     + model_c.coef[2] * model_c_data.vehicle_miles_mil
                                     + model_c.coef[3] * model_c_data.refinery_utilization_pct
                                     + model_c.coef[4] * model_c_data.us_population_thous
                                     + model_c.coef[5] * model_c_data.initial_jobless_claims)

score_c = evaluate(model_c_data)
score_c

In [ ]:
show_vif(model_c_data, model_c_predictors)

The VIFs are all under 5, so this is the only model so far whose coefficients are structurally
readable. But look at what it says about gas price: the coefficient is **+0.36 with a pval of
0.99**. On the 1991-2019 data this same variable came out at -58 with a pval of 0.006, which we
read as clean support for the hypothesis that higher prices reduce consumption.

Nothing about the variable changed. We added five years of data and the effect vanished. That
is not a result you can report either way yet - it is a signal that something in the model is
mis-specified, and we come back to it below.

Jobless claims is now strongly significant and negative, which is at least the direction you
would expect, though 2020 alone could be driving it.

## Model D - six predictors

On the 1991-2019 data, this combination had the highest adjusted r2 of any six-predictor
model. It is worth carrying forward to see whether that still holds.

In [ ]:
model_d_predictors = ['wti_spot_usd_per_bbl', 'brent_spot_usd_per_bbl',
                      'regular_gas_usd_per_gal', 'vehicle_miles_mil',
                      'gdp_saar_bil_usd', 'us_population_thous']

model_d_data = training[['Demand', 'Date'] + model_d_predictors].dropna().copy()

model_d = pg.linear_regression(X = model_d_data[model_d_predictors], y = model_d_data.Demand)
print(model_d)

In [ ]:
model_d_data["calculated_demand"] = (model_d.coef[0]
                                     + model_d.coef[1] * model_d_data.wti_spot_usd_per_bbl
                                     + model_d.coef[2] * model_d_data.brent_spot_usd_per_bbl
                                     + model_d.coef[3] * model_d_data.regular_gas_usd_per_gal
                                     + model_d.coef[4] * model_d_data.vehicle_miles_mil
                                     + model_d.coef[5] * model_d_data.gdp_saar_bil_usd
                                     + model_d.coef[6] * model_d_data.us_population_thous)

score_d = evaluate(model_d_data)
score_d

In [ ]:
show_vif(model_d_data, model_d_predictors)

The coefficient pattern is the same pathology as before, and worse for being predictable:

- **wti is about +31 and brent about -37**, though the two measure nearly the same thing and
  correlate at 0.99.
- **Gas price is about +445.** In Model C the same variable was statistically zero. Neither
  number is a price elasticity; both are artifacts of what else is in the equation.
- **GDP is negative again**, for the reason seen in Model B.

Five of the six VIFs are far above the threshold. A coefficient that changes sign and
magnitude depending on its neighbours is not measuring a real effect.

Let's also check whether this is still the best six-predictor combination on the new data,
rather than assuming it.

In [ ]:
well_populated = ['initial_jobless_claims', 'wti_spot_usd_per_bbl', 'brent_spot_usd_per_bbl',
                  'regular_gas_usd_per_gal', 'refinery_utilization_pct', 'rotary_rig_count',
                  'vehicle_sales_saar_mil', 'pce_saar_bil_usd', 'vehicle_miles_mil',
                  'gdp_saar_bil_usd', 'consumer_credit_mil_usd', 'smoking_adults_pct',
                  'us_population_thous']

search_data = training[['Demand'] + well_populated].dropna()

results = []
for combo in itertools.combinations(well_populated, 6):
    fitted = sm.OLS(search_data.Demand, sm.add_constant(search_data[list(combo)])).fit()
    results.append((fitted.rsquared_adj, combo))
results.sort(reverse = True)

print("Best three six-predictor combinations on this data:")
for r2, combo in results[:3]:
    print(f"  {r2:.4f}  {', '.join(combo)}")

rank = next(i for i, (r2, combo) in enumerate(results)
            if set(combo) == set(model_d_predictors))
print(f"\nModel D now ranks {rank + 1} of {len(results)} (adj_r2 {results[rank][0]:.4f})")

Model D has dropped from first to somewhere in the middle of the pack. Its status as "the
best six-predictor model" was a fact about the 1991-2019 sample, not a durable property, and
carrying that claim forward without rechecking it would have been wrong. Exhaustive searches
over 1,716 combinations find whatever fits the sample in front of them.

## Model E - thirteen predictors

Every predictor with data for at least 95% of the training weeks.

In [ ]:
coverage = training[well_populated].notna().mean().sort_values()
print("Lowest coverage among the thirteen:")
print(coverage.head(3).round(3).to_string())
print("\nNext excluded candidate:",
      round(training.regulated_afv_acquisitions.notna().mean(), 3))

model_e_predictors = well_populated
model_e_data = training[['Demand', 'Date'] + model_e_predictors].dropna().copy()

model_e = pg.linear_regression(X = model_e_data[model_e_predictors], y = model_e_data.Demand)
print()
print(model_e)

Rather than typing out thirteen terms by hand, loop through the coefficients. This produces
the same result as writing the equation out.

In [ ]:
model_e_data["calculated_demand"] = model_e.coef[0]

for i, predictor in enumerate(model_e_predictors):
    model_e_data["calculated_demand"] += model_e.coef[i + 1] * model_e_data[predictor]

score_e = evaluate(model_e_data)
score_e

Model E includes `smoking_adults_pct` with a positive coefficient at p < 0.001. Nobody thinks
smoking rates drive gasoline demand. It is a slow downward trend over thirty years that lines
up with other slow trends, and with thirteen predictors a few such coincidences are expected.
Statistical significance is not evidence of a mechanism.

There is also a numerical problem in the table above that is easy to miss. Look at the
Intercept row: pingouin reports a standard error near 0.1 on a coefficient in the thousands,
giving an absurd T statistic. That is not a real result. Model E's design matrix is so
ill-conditioned that the pseudo-inverse pingouin uses for standard errors loses precision.
The coefficients are fine; the inference around them is not. Refit with statsmodels and
compare.

In [ ]:
model_e_sm = sm.OLS(model_e_data.Demand,
                    sm.add_constant(model_e_data[model_e_predictors])).fit()

print("Condition number:", f"{model_e_sm.condition_number:,.0f}")
print("(above ~30 indicates collinearity; above 1e6 the standard errors")
print(" should not be trusted without a cross-check)")
print()
print("Intercept, pingouin   : coef", round(model_e.coef[0], 2),
      " se", round(model_e.se[0], 2), " pval", f"{model_e.pval[0]:.2e}")
print("Intercept, statsmodels: coef", round(model_e_sm.params['const'], 2),
      " se", round(model_e_sm.bse['const'], 2), " pval", f"{model_e_sm.pvalues['const']:.2e}")

Same coefficient, standard error several thousand times larger. The statsmodels figure is the
correct one; it uses a QR decomposition, which is numerically stable here where the
pseudo-inverse is not. pingouin's `linear_regression` is fine for the small, well-conditioned
models in this notebook, but Model E is past the point where it can be trusted for inference.

## Compare the models

Fit statistics are only comparable across models fitted on the same rows. Because each model
calls `.dropna()` on a different column subset, that is not guaranteed, so we check rather
than assume.

In [ ]:
comparison = pd.DataFrame({
    "model":  ["A (2)", "B (3)", "C (5)", "D (6)", "E (13)"],
    "n":      [score_a["n"], score_b["n"], score_c["n"], score_d["n"], score_e["n"]],
    "r2":     [model_a.r2[0], model_b.r2[0], model_c.r2[0], model_d.r2[0], model_e.r2[0]],
    "adj_r2": [model_a.adj_r2[0], model_b.adj_r2[0], model_c.adj_r2[0],
               model_d.adj_r2[0], model_e.adj_r2[0]],
    "RMSE":   [score_a["RMSE"], score_b["RMSE"], score_c["RMSE"],
               score_d["RMSE"], score_e["RMSE"]],
    "MAE":    [score_a["MAE"], score_b["MAE"], score_c["MAE"],
               score_d["MAE"], score_e["MAE"]]})

assert comparison.n.nunique() == 1, (
    "Models were fitted on different samples - r2 and RMSE are not comparable:\n"
    + comparison[["model", "n"]].to_string(index = False))

comparison.round(4)

All five land on the same 1,713 rows, so the comparison is valid. That is not luck to rely
on: `us_population_thous` has the most missing values and appears in all five models, so it
sets the sample every time. Drop it from a future model and the assertion fires instead of
letting us compare numbers that are not comparable.

r2 always rises when you add predictors, so it cannot tell you whether the extra variables
earn their place; adj_r2 penalises each addition and is the fairer comparison. Model E wins,
as it did before.

But every one of these models is worse than its 1991-2019 counterpart, and Model C's gas-price
coefficient went to zero. Before choosing among them we should find out why.

## Diagnosing the problem: are the residuals stable over time?

A regression assumes one relationship holds across the whole sample. If it does not, the
residuals will not scatter randomly around zero; they will drift. Plotting residuals against
date is the cheapest way to find out, and it is a check worth running on any model fitted to
a time series.

In [ ]:
model_c_data["error"] = model_c_data.Demand - model_c_data.calculated_demand

fig, ax = plt.subplots(figsize = (13, 4.5))
ax.scatter(model_c_data.Date, model_c_data.error, s = 4, alpha = 0.5, color = "#2b6cb0")
ax.axhline(0, color = "black", linewidth = 1)
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-01-01"),
           color = "crimson", alpha = 0.12)
ax.set_title("Model C residuals over time (actual minus estimated)")
ax.set_ylabel("residual")
plt.show()

model_c_data["era"] = np.select(
    [model_c_data.Date < "2020-03-01", model_c_data.Date < "2021-01-01"],
    ["pre-2020", "2020"], default = "2021+")

print(model_c_data.groupby("era").error.agg(["count", "mean", "std"]).round(1).to_string())

There it is. The residuals are not centred on zero within each period:

- **pre-2020**: mean residual about +65
- **2020**: about -203
- **2021 onward**: about -414

From 2021 the model over-estimates demand by roughly 400 units in almost every week. That is
a systematic bias, not noise, and it is close to the size of the whole MAE. The single fitted
line is splitting the difference between two eras, sitting too low before 2020 and too high
after.

This also explains the vanished gas-price coefficient. Post-2020 the model needs to come
down, and gas prices happened to be high in 2022-23; the fitted line uses whatever it can to
absorb the level shift, and the price term gets pulled away from its real value in the
process.

### A formal test for the break

The residual plot is suggestive. A Chow test makes it decisive: it compares the sum of
squared residuals from one pooled regression against two separate regressions split at a
candidate break date. If splitting explains substantially more, the pooled model is
mis-specified.

In [ ]:
def chow_test(data, preds, break_date):
    def ssr(df):
        return sm.OLS(df.Demand, sm.add_constant(df[preds])).fit().ssr

    before, after = data[data.Date < break_date], data[data.Date >= break_date]
    k = len(preds) + 1
    pooled, split = ssr(data), ssr(before) + ssr(after)
    f_stat = ((pooled - split) / k) / (split / (len(data) - 2 * k))
    p_value = 1 - stats.f.cdf(f_stat, k, len(data) - 2 * k)
    return {"break": break_date, "n_before": len(before), "n_after": len(after),
            "F": round(f_stat, 2), "p": p_value}

chow = chow_test(model_c_data, model_c_predictors, "2020-03-01")
print(chow)

F = 86.6 with a p-value indistinguishable from zero. We reject the null that the same
coefficients apply on both sides of March 2020. The relationship genuinely changed, and no
amount of adding predictors to a pooled model will fix a structural break.

## Model F - letting the model know about the break

The simplest correction is to give the regression two indicator variables: one for the 2020
disruption, one for the period after it. Each gets its own coefficient, so the model can
shift its level for those eras instead of distorting the real predictors to compensate.

An indicator (or dummy) variable is just a column of 0s and 1s. Its coefficient is the
average shift in demand for the weeks flagged with a 1, holding everything else constant.

In [ ]:
training["covid_2020"] = ((training.Date >= "2020-03-01") &
                          (training.Date < "2021-01-01")).astype(int)
training["post_2020"] = (training.Date >= "2021-01-01").astype(int)

print("weeks flagged covid_2020:", training.covid_2020.sum())
print("weeks flagged post_2020 :", training.post_2020.sum())

model_f_predictors = model_c_predictors + ['covid_2020', 'post_2020']
model_f_data = training[['Demand', 'Date'] + model_f_predictors].dropna().copy()

model_f = pg.linear_regression(X = model_f_data[model_f_predictors], y = model_f_data.Demand)
print()
print(model_f)

In [ ]:
model_f_data["calculated_demand"] = model_f.coef[0]
for i, predictor in enumerate(model_f_predictors):
    model_f_data["calculated_demand"] += model_f.coef[i + 1] * model_f_data[predictor]

score_f = evaluate(model_f_data)
print(score_f)
print()
print(show_vif(model_f_data, model_f_predictors).to_string(index = False))

Adjusted r2 rises from 0.545 to 0.626 by adding two columns of 0s and 1s, and the VIFs stay
low. Both indicators are strongly significant:

- **covid_2020 is about -630.** During March-December 2020, demand ran 630 units below what
  the predictors alone imply.
- **post_2020 is about -707.** From 2021 onward, demand runs about 707 units below the old
  relationship, and it has not recovered in four years of data.

That second number is the substantive finding in this notebook. It is not a pandemic dip; it
is a permanent downward shift in the level of gasoline demand, most plausibly reflecting
changes in commuting behaviour, vehicle efficiency, and fleet electrification that arrived
alongside the pandemic and stayed.

Note that gas price is *still* insignificant even with the break handled. That is now an
honest result rather than a symptom: once driving activity is in the model, weekly price
adds little on top of it, because price acts on demand largely by changing how much people
drive.

### Confirming it another way: fit the eras separately

Dummies allow the level to shift but force every slope to stay the same. Fitting each period
on its own lets everything move, which is a stronger check.

In [ ]:
periods = {
    "1991 to Feb 2020": model_c_data[model_c_data.Date < "2020-03-01"],
    "2021 onward":      model_c_data[model_c_data.Date >= "2021-01-01"]}

rows = []
for label, subset in periods.items():
    fitted = sm.OLS(subset.Demand, sm.add_constant(subset[model_c_predictors])).fit()
    row = {"period": label, "n": len(subset), "adj_r2": round(fitted.rsquared_adj, 4)}
    row.update({p: round(fitted.params[p], 4) for p in model_c_predictors})
    rows.append(row)

pd.DataFrame(rows).set_index("period").T

The pre-2020 fit reaches an adjusted r2 of 0.667 - essentially what this specification
achieved on the old file, which makes sense, since it is nearly the old sample. The 2021+ fit
manages 0.05.

That gap is the real message. These predictors described gasoline demand well for thirty
years and describe it poorly now. With only 209 post-2020 weeks it is too early to say
whether a stable new relationship has formed or the series is still settling, but reporting
the pooled model's numbers as though nothing happened would be misleading.

## Which model to use

It depends on the question, and after the diagnostics the answer differs from what the
comparison table alone suggests.

**To forecast the holdout period**, use Model F or a variant of it. The holdout weeks are all
in 2025-26, which sits inside the post-2020 regime, so `post_2020` will be 1 for every one of
them. A model without that term will over-forecast by roughly 700 units per week, in the same
direction every week - the worst kind of forecast error, because it never averages out.

**To explain what drives demand**, use Model F for the same reason. It is the only model here
that is both structurally sound (all VIFs under 7) and not systematically biased by era.

**Do not use Model D or E for explanation** despite their fit. Their coefficients are
collinearity artifacts, and Model E's standard errors are numerically unreliable on top of
that.

One caution about Model F: an indicator variable records *that* the level shifted, not *why*.
It will keep applying a -707 adjustment to every future week indefinitely. That is a
reasonable assumption over a short horizon and a poor one over a long one, and it should be
revisited as more post-2020 data arrives.

## Build the Excel formulas

The spreadsheet column letters are fixed by the layout of the Target sheet, so we can map
each predictor to its letter and print a formula to paste into Excel. The mapping is verified
against the sheet header rather than trusted, so a re-ordered column would raise an error
instead of silently producing wrong numbers.

The two indicator variables are not columns in the sheet, so they are written as inline date
tests against the Date column (B).

In [ ]:
header = pd.read_excel(DATA_FILE, sheet_name = 'Target', nrows = 0)
letters = {name: (chr(65 + i) if i < 26 else 'A' + chr(65 + i - 26))
           for i, name in enumerate(header.columns)}

expected = {'initial_jobless_claims': 'E', 'wti_spot_usd_per_bbl': 'G',
            'brent_spot_usd_per_bbl': 'H', 'regular_gas_usd_per_gal': 'I',
            'gdp_saar_bil_usd': 'J', 'rotary_rig_count': 'L',
            'vehicle_sales_saar_mil': 'M', 'vehicle_miles_mil': 'N',
            'pce_saar_bil_usd': 'O', 'consumer_credit_mil_usd': 'P',
            'smoking_adults_pct': 'U', 'refinery_utilization_pct': 'V',
            'us_population_thous': 'AA'}

for predictor, letter in expected.items():
    assert letters[predictor] == letter, \
        f"{predictor} is in column {letters[predictor]}, not {letter}"
print("All 13 column letters verified against the sheet header.")

excel_terms = {name: letters[name] + "2" for name in expected}
excel_terms["covid_2020"] = 'IF(AND($B2>=DATE(2020,3,1),$B2<DATE(2021,1,1)),1,0)'
excel_terms["post_2020"]  = 'IF($B2>=DATE(2021,1,1),1,0)'

In [ ]:
all_models = {"A": (model_a, model_a_predictors), "B": (model_b, model_b_predictors),
              "C": (model_c, model_c_predictors), "D": (model_d, model_d_predictors),
              "E": (model_e, model_e_predictors), "F": (model_f, model_f_predictors)}

for name, (model, preds) in all_models.items():
    formula = "=" + format(model.coef[0], ".8f")
    for i, predictor in enumerate(preds):
        formula += "+" + format(model.coef[i + 1], ".8f") + "*" + excel_terms[predictor]
    formula = formula.replace("+-", "-")
    print("Model", name)
    print(formula)
    print()

Paste any of these into row 2 of an empty column on the Target sheet and fill down. The
references are relative, so row 2 becomes row 3 and so on automatically. Model F is the one
to use on the holdout rows.

One caution: Excel reads a blank cell as a zero, so on the early rows where population or GDP
are missing the formula returns a number instead of an error. Those values are not real
estimates. To make them visibly wrong rather than quietly wrong, wrap the formula so it
blanks out on incomplete rows:

```
=IF(COUNTBLANK(N2)+COUNTBLANK(AA2)>0, "", <formula here>)
```

## Appendix - why we did not fill in the missing values

A tempting shortcut for the missing population and GDP weeks is to forward-fill and then
backward-fill so no rows are lost. It is worth seeing what that actually does.

Forward-filling is defensible for GDP, which is reported quarterly, so carrying the last
published figure through the weeks is close to how the series behaves. Backward-filling is a
different matter. The population gaps are all at the *start* of the sample, so there is
nothing earlier to carry forward, and `.bfill()` reaches backwards and stamps the first
observed value onto every one of those weeks - a flat line where the real series was rising.

In [ ]:
appendix_predictors = ['vehicle_miles_mil', 'us_population_thous', 'gdp_saar_bil_usd',
                       'regular_gas_usd_per_gal', 'wti_spot_usd_per_bbl',
                       'refinery_utilization_pct', 'initial_jobless_claims']

def fit_ols(X, y, label):
    data = pd.concat([y, X], axis = 1).dropna()
    fitted = sm.OLS(data.Demand, sm.add_constant(data[appendix_predictors])).fit()
    return {"strategy": label, "n": len(data), "adj_r2": round(fitted.rsquared_adj, 4)}

raw = training[appendix_predictors]

print(pd.DataFrame([
    fit_ols(raw, training.Demand, "complete case (what we used)"),
    fit_ols(raw.ffill(), training.Demand, "ffill only"),
    fit_ols(raw.ffill().bfill(), training.Demand, "ffill then bfill")
]).to_string(index = False))

missing_mask = raw.us_population_thous.isna()
print("\nWeeks with fabricated population:", missing_mask.sum())
print("All assigned the value:",
      raw.us_population_thous.ffill().bfill()[missing_mask].unique())

The backward-filled model reports a higher adjusted r2 than the honest fit, and that gain is
manufactured: 56 rows of invented data, every one carrying the same population figure, added
to a model where population is a leading predictor. A flat predictor column paired with real
variation in demand is easy for OLS to fit around, and it makes the model look better than
it is.

There is a second problem the r2 does not reveal. OLS treats every row as an independent
observation, so filled-in values count as evidence at full strength. Standard errors come out
too small and p-values too optimistic, because the model cannot know that some of its
observations were made up.

Complete-case analysis costs 56 weeks out of 1,769 - about 3% - and they are all at the very
beginning of a thirty-four-year series. That is a cheap price for numbers we can defend.

## Summary

- The training data now runs 1991-02-08 to 2024-12-27 (1,769 weeks), with 87 holdout weeks
  through August 2026.
- Every model in the original specification set fits worse on this data than on 1991-2019,
  and Model C's gas-price coefficient collapses to zero.
- The cause is a structural break at March 2020, confirmed by residual drift and a Chow test
  (F = 86.6, p < 0.001). Demand sits roughly 700 units below the old relationship from 2021
  onward and has not recovered.
- Adding two indicator variables (Model F) lifts adjusted r2 from 0.545 to 0.626, removes the
  systematic bias, and keeps every VIF under 7.
- Models D and E fit better still but their coefficients are collinearity artifacts; Model E's
  standard errors are also numerically unreliable at a condition number above 1e8.
- Model D is no longer the best six-predictor combination on this data, which is a reminder
  that exhaustive-search results describe the sample, not the world.